# Физически-информированные нейронные сети

**Проект «ИИ для учёных».** Практический блокнот к странице алгоритма «Физически-информированные нейронные сети».

## Что делает метод

Физически-информированная нейронная сеть (PINN) обучается не только на измерениях, но и на самих уравнениях процесса. Сеть представляет искомую функцию (поле температуры, скорости, концентрации), а в функцию потерь добавляется невязка дифференциального уравнения. В результате решение согласовано с физикой даже там, где измерений мало или нет совсем.

Это полезно, когда классический численный решатель труден (сложная геометрия, неизвестные параметры), а данных недостаточно для обычной регрессии.

В этом блокноте мы решим PINN простое обыкновенное дифференциальное уравнение и сравним результат с точным решением. Сеть — небольшая, на базовом PyTorch; обучение занимает секунды на CPU. Расчётное время прохождения — около пяти минут.

## Интуиция за методом

Представьте, что нужно восстановить температурный профиль вдоль стержня, но термометров у вас только три — в начале, конце и в середине. Данных мало, чтобы подогнать произвольную кривую. Но вы знаете закон теплопроводности — дифференциальное уравнение, которому этот профиль обязан подчиняться в каждой точке. Тогда правильная стратегия: искать не любую кривую через три точки, а только такую, которая одновременно проходит близко к замерам **и** не нарушает физический закон.

Именно это делает физически-информированная нейронная сеть (PINN):
- Нейросеть представляет искомую функцию (температуру, скорость, концентрацию) — непрерывную, дифференцируемую.
- В функцию потерь добавляется **невязка** дифференциального уравнения — насколько предсказание сети нарушает закон физики.
- Обучение минимизирует одновременно отклонение от данных и нарушение уравнения.

**Невязка** — разность между левой и правой частями дифференциального уравнения, вычисленная для текущего предсказания сети. Если невязка равна нулю, сеть точно удовлетворяет уравнению.

**Точки коллокации** — точки в области решения, в которых проверяется выполнение уравнения. Их выбирают достаточно плотно, чтобы охватить всю область.

**Автоматическое дифференцирование** — встроенный в PyTorch механизм точного вычисления производных: производная берётся не численно (конечными разностями), а аналитически по вычислительному графу. Это позволяет без усилий вычислить du/dt прямо из предсказания сети.

## 1. Установка библиотек

In [ ]:
%pip install -q torch numpy matplotlib

## 2. Единый стиль графиков

Графики оформляются в едином визуальном стиле сайта «ИИ для учёных». Ниже встроено содержимое стилевого шаблона `notebooks/viz_style.py`.

In [ ]:
# Единый стилевой шаблон графиков проекта «ИИ для учёных».
# Значения синхронизированы с токенами темы сайта (styles/tokens.css).
VIZ_TOKENS = {
    "background": "#ffffff",
    "node_fill":  "#eef1f6",
    "node_text":  "#0e1726",
    "edge":       "#4d5e78",
    "grid":       "#dce2ec",
    "series":     ["#2563eb", "#0d9488", "#b45309", "#4d5e78"],
}
VIZ = VIZ_TOKENS


def apply_viz_style():
    """Настраивает matplotlib под единый стиль сайта."""
    import matplotlib as mpl
    from cycler import cycler
    t = VIZ_TOKENS
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "semibold",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.facecolor": t["background"], "axes.facecolor": t["background"],
        "savefig.facecolor": t["background"], "figure.dpi": 110, "savefig.dpi": 150,
        "axes.prop_cycle": cycler(color=t["series"]),
        "text.color": t["node_text"], "axes.labelcolor": t["node_text"],
        "axes.titlecolor": t["node_text"], "axes.edgecolor": t["edge"],
        "xtick.color": t["edge"], "ytick.color": t["edge"], "axes.linewidth": 1.2,
        "axes.grid": True, "grid.color": t["grid"], "grid.linewidth": 1.0,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 2.0, "lines.markersize": 6, "patch.linewidth": 1.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False,
    })


def get_palette(n=None):
    """Возвращает список цветов рядов из токенов темы."""
    series = VIZ_TOKENS["series"]
    if n is None:
        return list(series)
    return [series[i % len(series)] for i in range(n)]


apply_viz_style()
print("Единый стиль графиков подключён.")

## 3. Постановка задачи: уравнение остывания тела по Ньютону

Решаем простое, но физически осмысленное уравнение:

$$\frac{du}{dt} = -k \cdot u(t), \qquad u(0) = u_0.$$

Здесь $u(t)$ — температура тела сверх температуры среды; $k > 0$ — коэффициент теплоотдачи; $u_0$ — начальная температура.

**Точное решение** известно аналитически: $u(t) = u_0 \cdot e^{-kt}$ — экспоненциальное остывание. Мы используем его **только для проверки** в конце. Сама PINN о нём не знает: она работает только с уравнением и начальным условием.

Обратите внимание на `t_collocation` — 60 равномерно расположенных точек на отрезке [0, T]. Это точки коллокации: в них будет проверяться выполнение уравнения.

In [ ]:
import numpy as np
import torch

torch.manual_seed(0)
np.random.seed(0)

k = 1.5            # коэффициент остывания
u0 = 2.0           # начальная температура
T = 3.0            # горизонт моделирования

def exact(t):
    """Точное решение - только для контроля качества."""
    return u0 * np.exp(-k * t)

# Точки коллокации, в которых проверяется выполнение уравнения.
t_collocation = torch.linspace(0, T, 60).reshape(-1, 1)
print(f"Точек коллокации: {len(t_collocation)}")

## 4. Применение метода

### Шаг 1. Нейросеть как представление решения

Определяем небольшую полносвязную нейросеть: вход — момент времени $t$ (одно число), выход — предсказанная температура $u(t)$ (одно число). Промежуточные слои используют функцию активации `tanh` (гиперболический тангенс) — она гладкая и дифференцируемая, что позволяет корректно вычислять производные автоматическим дифференцированием.

Число параметров (весов) сети невелико — несколько сотен. Это намеренно: задача простая, и большая сеть здесь ничего не даст.

Ячейка выводит общее число обучаемых параметров.

In [ ]:
import torch.nn as nn

class PINN(nn.Module):
    """Малая сеть, представляющая искомую функцию u(t)."""
    def __init__(self, hidden=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, 1),
        )

    def forward(self, t):
        return self.net(t)


model = PINN()
n_params = sum(p.numel() for p in model.parameters())
print(f"Параметров в сети: {n_params}")

### Шаг 2. Функция потерь с физическим условием

Суммарные потери складываются из двух частей:

**Часть 1 — невязка уравнения** (в точках коллокации):
- Сеть предсказывает $u(t)$ в каждой точке коллокации.
- Автоматическим дифференцированием вычисляется $du/dt$ — производная предсказания по времени.
- Невязка: $du/dt + k \cdot u$ должна быть равна нулю. Среднее значение квадрата невязки — это первая часть потерь.

**Часть 2 — начальное условие**:
- Сеть предсказывает $u(0)$.
- Отклонение от заданного $u_0$ — вторая часть потерь.

Коэффициент `10.0` перед начальным условием — вес: он говорит «ошибка в начальном условии для нас в 10 раз важнее, чем ошибка в уравнении». Это стандартный способ балансировки частей потерь в PINN.

In [ ]:
def physics_loss(model, t):
    """Невязка дифференциального уравнения в точках коллокации."""
    t = t.clone().requires_grad_(True)
    u = model(t)
    # du/dt средствами автоматического дифференцирования.
    du_dt = torch.autograd.grad(u, t, grad_outputs=torch.ones_like(u),
                                create_graph=True)[0]
    residual = du_dt + k * u            # должно быть равно нулю
    return torch.mean(residual ** 2)


def initial_loss(model):
    """Отклонение от начального условия u(0) = u0."""
    u_pred = model(torch.zeros(1, 1))
    return torch.mean((u_pred - u0) ** 2)

### Шаг 3. Обучение

Запускаем 1500 итераций оптимизатора Adam — стандартного метода обновления весов нейросети. Каждые 300 итераций выводятся текущие потери. Ожидаем, что они упадут на несколько порядков.

Обратите внимание: сеть не видит ни одного значения $u(t)$ из точного решения — только уравнение и начальное условие. Если обучение пройдёт успешно, сеть «выведет» закон экспоненциального остывания самостоятельно.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
n_epochs = 1500
history = []

for epoch in range(n_epochs):
    optimizer.zero_grad()
    loss = physics_loss(model, t_collocation) + 10.0 * initial_loss(model)
    loss.backward()
    optimizer.step()
    history.append(loss.item())
    if (epoch + 1) % 300 == 0:
        print(f"Эпоха {epoch + 1:4d}   суммарные потери = {loss.item():.2e}")

print("Обучение завершено.")

### Шаг 4. Сравнение с точным решением

Теперь можно сравнить: что выдаёт обученная сеть, и что является правильным ответом. Строим оба на одном графике, а также кривую сходимости потерь.

In [ ]:
import matplotlib.pyplot as plt

t_grid = torch.linspace(0, T, 200).reshape(-1, 1)
with torch.no_grad():
    u_pinn = model(t_grid).numpy().ravel()
t_np = t_grid.numpy().ravel()
u_true = exact(t_np)

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.2))

axes[0].plot(t_np, u_true, color=VIZ["series"][3], linewidth=3,
             label="Точное решение")
axes[0].plot(t_np, u_pinn, color=VIZ["series"][0], linestyle="--",
             label="Решение PINN")
axes[0].set_title("Решение уравнения остывания")
axes[0].set_xlabel("Время t")
axes[0].set_ylabel("Температура u(t)")
axes[0].legend(loc="upper right")

axes[1].plot(history, color=VIZ["series"][1])
axes[1].set_yscale("log")
axes[1].set_title("Сходимость обучения")
axes[1].set_xlabel("Эпоха")
axes[1].set_ylabel("Суммарные потери (лог. шкала)")

fig.tight_layout()
plt.show()

mae = np.mean(np.abs(u_pinn - u_true))
print(f"Средняя абсолютная ошибка относительно точного решения: {mae:.4f}")

**Как читать эти графики.**

Левый — решение уравнения:
- Серая сплошная линия — точное аналитическое решение $u_0 e^{-kt}$ (правильный ответ, вычисленный «в лоб»).
- Синяя пунктирная линия — решение PINN (сеть не знала правильного ответа, только уравнение).
- Чем ближе линии друг к другу, тем лучше сеть решила уравнение. Если они почти совпадают — это наглядное подтверждение того, что физическое ограничение в функции потерь действительно работает.
- MAE (средняя абсолютная ошибка), выводимая ниже, — количественная оценка расхождения.

Правый — сходимость обучения (логарифмическая шкала):
- Ось Y — в логарифмическом масштабе: падение с 1e+0 до 1e-4 означает уменьшение потерь в 10 000 раз.
- Гладкое убывание без «плато» — признак успешного обучения.
- Если кривая застыла на высоком уровне — нужно больше итераций, другой шаг обучения или больше точек коллокации.

## 5. Интерпретация результата

- **Левый график**: кривая PINN практически совпадает с точным решением, хотя сеть не видела ни одного значения $u(t)$ — она опиралась только на уравнение и начальное условие. В этом и состоит суть метода.
- **Правый график**: потери падают на несколько порядков. Невязка уравнения, близкая к нулю, означает, что сеть выдаёт функцию, физически согласованную.
- **Вес начального условия** (множитель 10) балансирует две части потерь. Если условие выполняется плохо, вес увеличивают; это типичная настройка PINN.
- Точки коллокации задают, где проверяется уравнение. Их нужно достаточно, чтобы покрыть всю область решения.

Помните: PINN особенно ценны, когда к уравнению добавляются немногочисленные реальные измерения — тогда сеть одновременно решает уравнение и подгоняется под данные, в том числе восстанавливая неизвестные коэффициенты.

## 6. Поэкспериментируйте сами

Измените один параметр, повторно запустите ячейки разделов 3 и 4 и сравните результат.

**Что менять и что наблюдать:**
- Измените `k = 1.5` на `k = 0.3` (медленное остывание) или `k = 4.0` (быстрое). Справляется ли PINN в обоих случаях?
- Уменьшите `t_collocation` с 60 точек до 10: `torch.linspace(0, T, 10)`. Когда точек слишком мало, сеть хуже выполняет уравнение — как изменится кривая PINN?
- Попробуйте убрать начальное условие (поставьте коэффициент `0.0` вместо `10.0` в строке `loss = ...`). Сойдётся ли сеть к правильному решению без фиксации начальной точки?
- Добавьте 5 «измерений»: создайте тензор `t_data` и `u_data` с несколькими точками точного решения и добавьте слагаемое `data_loss` из шаблона в разделе 6. Ускоряет ли это сходимость?

In [ ]:
# --- Шаблон слагаемого потерь по измерениям ---
# t_data = torch.tensor([[0.5], [1.0], [2.0]])      # моменты измерений
# u_data = torch.tensor([[1.1], [0.5], [0.1]])      # измеренные значения
#
# def data_loss(model):
#     return torch.mean((model(t_data) - u_data) ** 2)
#
# В цикле обучения:
# loss = physics_loss(model, t_collocation) \
#      + 10.0 * initial_loss(model) + 5.0 * data_loss(model)

## Краткие выводы

- PINN объединяет нейросеть с физическими уравнениями: сеть представляет решение, а уравнение входит в функцию потерь как невязка.
- Ключевой результат: сеть находит правильное решение, не видя ни одного значения из него — только уравнение и начальное условие. Физический закон заменяет данные.
- Невязка дифференциального уравнения вычисляется автоматическим дифференцированием — точно, без конечных разностей.
- Балансировка весов частей потерь (уравнение, начальное условие, данные) — ключевой практический вопрос при настройке PINN.
- Метод особенно ценен для обратных задач: можно добавить неизвестный коэффициент (например, k) как обучаемый параметр и восстановить его из измерений вместе с решением.

## Три вопроса для самопроверки

Ответьте до того, как раскроете подсказку, — это проверяет, что метод понят, а не просто запущен.

1. Вы увеличиваете вес начального условия с 10 до 100. Кривая потерь снижается быстрее в первые 200 итераций, но затем выходит на плато выше, чем при весе 10, а невязка уравнения остаётся большой. Объясните, что произошло, и как правильно балансировать слагаемые функции потерь.
2. Вы добавляете к задаче пять реальных измерений $u(t_i)$ и хотите заодно восстановить неизвестный коэффициент $k$ — это обратная задача. Опишите, как именно нужно изменить код блокнота (какие переменные, функции, слагаемые потерь) и что принципиально отличает эту постановку от прямой задачи.
3. Ваша PINN хорошо сходится для $k = 1.5$, но при $k = 8.0$ (быстрое остывание) потери остаются высокими даже после 5000 итераций. Назовите две причины, связанные со свойствами уравнения, и предложите конкретные меры.

<details>
<summary>Показать ориентиры для ответов</summary>

1. Слишком большой вес начального условия «перетягивает» оптимизацию: сеть сначала почти идеально выполняет $u(0) = u_0$, но при этом градиенты, связанные с невязкой уравнения, становятся относительно малы и не двигают веса в нужном направлении. Итог — локальный минимум, где начальное условие выполнено, а уравнение нет. Балансировку ведут адаптивно: отслеживают соотношение величин двух слагаемых и подбирают веса так, чтобы они были одного порядка, или используют алгоритмы автоматической балансировки (NTK-взвешивание, self-adaptive PINN).
2. Для обратной задачи $k$ объявляется как `k = torch.nn.Parameter(torch.tensor(1.0))` и включается в `model.parameters()`. Добавляется слагаемое `data_loss` — среднеквадратичное отклонение предсказания сети от измерений. Суммарные потери: `physics_loss + weight_ic * initial_loss + weight_data * data_loss`. В отличие от прямой задачи, здесь оптимизируются одновременно веса сети и неизвестный параметр; это требует более осторожного выбора весов слагаемых и достаточного числа измерений для идентифицируемости $k$.
3. Две причины: (а) при большом $k$ функция $u(t)$ быстро убывает и имеет крутой градиент у $t=0$ — сети трудно аппроксимировать этот масштаб; (б) «спектральное смещение» (spectral bias) нейросетей: они предпочтительно выучивают низкочастотные компоненты, а быстроубывающая экспонента содержит высокие частоты. Меры: (а) сгустить точки коллокации вблизи $t=0$, например, выбирая их из логарифмически равномерного распределения; (б) применить масштабирование входа — подавать сети $t' = k \cdot t$ вместо $t$, чтобы нормализовать характерный масштаб решения.
</details>